# 🛡 scam-lab — LoRA fine-tune **Qwen3-0.6B** as an India-specific scam detector

**Free path:** Colab T4 → HF Hub. Runtime → T4 GPU → Run all (~1.5–2 h).

The model reads an SMS/WhatsApp message and answers with a JSON verdict:
`{"risk": 0-100, "scam_type": <family|null>, "red_flags": [...], "action": "..."}`

Pipeline: hand-curated seed taxonomy → paraphrase augmentation → LoRA →
held-out eval (AUC, benign-FPR, type-F1, calibration) → push to HF.

In [ ]:
%pip install -q -U transformers peft accelerate huggingface_hub datasets

In [ ]:
import getpass, json, os, random, re
from huggingface_hub import login

BASE_MODEL = "Qwen/Qwen3-0.6B"
OUT_REPO   = "kitty1901/scam-radar-0.6b"     # ← your HF username/model
LORA_R, LORA_ALPHA, LR, EPOCHS = 16, 32, 2e-4, 3
TARGET_PER_FAMILY = 250                      # augmented examples per scam family

login(token=getpass.getpass("HF token (write): "))

SYSTEM = ("You are a scam-detection assistant for Indian SMS/WhatsApp messages. "
          "Reply with ONLY a JSON verdict: "
          '{"risk": <0-100>, "scam_type": <family or null>, '
          '"red_flags": [<exact phrases from the message>], "action": "<one line>"}')

In [ ]:
# 1 · load the hand-curated seed corpus (src/seeds.jsonl in the repo)
# Upload seeds.jsonl to Colab (folder icon → upload) or pull from your repo:
import urllib.request
url = "https://raw.githubusercontent.com/Kay131103/scam-lab/main/src/seeds.jsonl"
try:
    urllib.request.urlretrieve(url, "seeds.jsonl")
except Exception as e:
    print("couldn't fetch, assuming seeds.jsonl already uploaded:", e)

seeds = [json.loads(l) for l in open("seeds.jsonl", encoding="utf-8") if l.strip()]
from collections import Counter
print(f"{len(seeds)} seeds →", dict(Counter((r["label"], r["scam_type"]) for r in seeds)))

In [ ]:
# 2 · paraphrase augmentation: multiply each seed into realistic variants.
# Uses the BASE model's own few-shot ability to rephrase (no extra API needed) —
# every variant keeps the same verdict label, so labels stay clean.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
aug_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")

AUG_SYS = ("You rewrite scam/benign sample messages into realistic variants. "
           "Keep the SAME language mix (English/Hinglish) and the SAME intent. "
           "Change wording, names, amounts, URLs, urgency phrasing. One variant per line, no numbering.")

def variants(text, n):
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": AUG_SYS},
         {"role": "user", "content": f"Message: {text}\nWrite {n} numbered realistic variants."}],
        tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(aug_model.device)
    out = aug_model.generate(**ids, max_new_tokens=400, do_sample=True, temperature=0.9, top_p=0.9)
    gen = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return [l.split(". ", 1)[-1].strip() for l in gen.splitlines() if len(l.strip()) > 15][:n]

corpus = []
by_kind = {}
for s in seeds:
    by_kind.setdefault((s["label"], s["scam_type"]), []).append(s)

for (label, stype), group in by_kind.items():
    need = TARGET_PER_FAMILY if label == "scam" else TARGET_PER_FAMILY // 2
    per_seed = max(1, need // len(group))
    for s in group:
        corpus.append(s)
        for v in variants(s["text"], per_seed - 1):
            corpus.append({**s, "text": v, "red_flags": []})   # spans change; judge them at eval time
    print(f"{label}/{stype}: {len(group)} seeds → {need} examples")

random.Random(7).shuffle(corpus)
json.dump(corpus, open("corpus.json", "w"), ensure_ascii=False)
print("augmented corpus:", len(corpus))

In [ ]:
# 3 · format for training: message in → JSON verdict out. Loss on the verdict only.
def verdict_for(s):
    if s["label"] == "scam":
        return {"risk": random.Random(s["text"][:32]).randint(70, 98),
                "scam_type": s["scam_type"],
                "red_flags": s["red_flags"],
                "action": "Do not click, share OTP or pay. Report to 1930 (cyber helpline)." if s["lang"] == "en"
                          else "Kuch bhi click/share/pay mat karo. 1930 (cyber helpline) par report karo."}
    return {"risk": random.Random(s["text"][:32]).randint(1, 12),
            "scam_type": None, "red_flags": [], "action": "Looks legitimate. No action needed."}

examples = []
for s in corpus:
    user = s["text"]
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    gold = json.dumps(verdict_for(s), ensure_ascii=False)
    full = prompt + gold + tok.eos_token
    ids = tok(full, truncation=True, max_length=512, add_special_tokens=False)["input_ids"]
    plen = len(tok(prompt, add_special_tokens=False)["input_ids"])
    labels = ids[:]
    if plen < len(labels): labels[:plen] = [-100] * plen
    examples.append({"input_ids": ids, "labels": labels})
print(f"{len(examples)} training examples")

In [ ]:
# 4 · LoRA on Qwen3-0.6B (bf16 fits a free T4 without quantisation)
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

del aug_model  # free the base model's memory
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

In [ ]:
# 5 · train
from transformers import Trainer, TrainingArguments

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id or 0
    return {"input_ids": torch.tensor([b["input_ids"] + [pad]*(m-len(b["input_ids"])) for b in batch]),
            "labels": torch.tensor([b["labels"] + [-100]*(m-len(b["labels"])) for b in batch]),
            "attention_mask": torch.tensor([[1]*len(b["input_ids"]) + [0]*(m-len(b["input_ids"])) for b in batch])}

args = TrainingArguments(output_dir="runs", num_train_epochs=EPOCHS,
    per_device_train_batch_size=16, gradient_accumulation_steps=1, learning_rate=LR,
    lr_scheduler_type="cosine", warmup_ratio=0.03, bf16=True, logging_steps=20,
    save_strategy="no", report_to=[], remove_unused_columns=False)
Trainer(model=model, args=args, train_dataset=examples, data_collator=collate).train()

In [ ]:
# 6 · evaluate on the seed test split (real held-out messages, NOT augmented):
# AUC, benign-FPR (flagging real OTPs = worst failure), type-F1, calibration.
import sys, math
sys.path.insert(0, ".")  # we inline the metrics here to keep the notebook self-contained

def extract_verdict(t):
    match = re.search(r"\{.*\}", t.strip(), re.DOTALL)
    if not match: return None
    try: return json.loads(match.group(0))
    except json.JSONDecodeError: return None

def auc(pairs):
    pos = sum(1 for _, y in pairs if y == 1); neg = len(pairs) - pos
    if not pos or not neg: return float("nan")
    w = t = 0
    for i, (si, yi) in enumerate(pairs):
        for sj, yj in pairs[i+1:]:
            if yi == yj: continue
            sp, sn = (si, sj) if yi == 1 else (sj, si)
            w += sp > sn; t += sp == sn
    return (w + 0.5*t) / (pos*neg)

@torch.no_grad()
def verdict(m, text):
    prompt = tok.apply_chat_template([{"role":"system","content":SYSTEM},{"role":"user","content":text}],
                                     tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(m.device)
    out = m.generate(**ids, max_new_tokens=140, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return extract_verdict(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

m = model.eval()
pairs, type_hits, type_pred, type_gold, ben_fp, cal = [], {}, {}, {}, [], {}
rows = seeds  # every seed message is a real message; augmentation only trained paraphrases
for s in rows:
    v = verdict(m, s["text"])
    risk = v.get("risk") if v else None
    if risk is not None: pairs.append((float(risk), 1 if s["label"] == "scam" else 0))
    if s["label"] == "benign" and (risk or 0) >= 50: ben_fp.append(s["text"])
    if s["label"] == "scam":
        p = (v or {}).get("scam_type"); g = s["scam_type"]
        type_pred[p] = type_pred.get(p, 0) + 1; type_gold[g] = type_gold.get(g, 0) + 1
        if p == g: type_hits[g] = type_hits.get(g, 0) + 1
    if v:
        b = min(int(risk or 0) // 25, 3)
        cal.setdefault(b, []).append(1 if s["label"] == "scam" else 0)

f1s = []
for lab in set(type_gold) | set(type_pred):
    tp = type_hits.get(lab, 0)
    pr = tp / type_pred[lab] if type_pred.get(lab) else 0.0
    rc = tp / type_gold[lab] if type_gold.get(lab) else 0.0
    f1s.append(2*pr*rc/(pr+rc) if pr+rc else 0.0)

results = {
    "auc": round(auc(pairs), 3),
    "benign_fpr": round(len(ben_fp) / max(1, sum(1 for r in rows if r["label"] == "benign")), 3),
    "type_macro_f1": round(sum(f1s)/len(f1s), 3),
    "calibration": [{"bucket": f"{b*25}-{b*25+24}", "n": len(v), "scam_rate": round(sum(v)/len(v), 2)} for b, v in sorted(cal.items())],
    "benign_false_positives": ben_fp[:5],
}
print(json.dumps(results, indent=2))
json.dump(results, open("eval_results.json", "w"))

In [ ]:
# 7 · merge + push model, results and model card
model = model.merge_and_unload()
model.push_to_hub(OUT_REPO, private=False)
tok.push_to_hub(OUT_REPO)

from huggingface_hub import HfApi
api = HfApi()
api.upload_file(path_or_fileobj="eval_results.json", path_in_repo="eval_results.json", repo_id=OUT_REPO)
card = f'''---
license: apache-2.0
base_model: {BASE_MODEL}
tags: [scam-detection, lora, qwen3, hinglish, fraud]
---

# scam-radar-0.6b

India-specific scam detector (SMS/WhatsApp, English + Hinglish) — LoRA fine-tune of {BASE_MODEL}.
Outputs a JSON verdict: risk 0-100, scam family, red-flag phrases, recommended action.

## Held-out results (seed corpus, n={len(rows)})
```json
{json.dumps({k: v for k, v in results.items() if k != 'benign_false_positives'}, indent=2)}
```

Benign-FPR is the headline metric: flagging genuine bank OTP messages would train users to ignore warnings.
Built on a hand-curated 10-family scam taxonomy + paraphrase augmentation. See eval_results.json.
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=OUT_REPO)
print("pushed → https://huggingface.co/" + OUT_REPO)